# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata

print(f"Dataset title: {getattr(meta, 'name', None)}")
print(f"Description: {getattr(meta, 'description', None)}")
print(f"Authors: {getattr(meta, 'author', None)}")
print(f"Keywords: {getattr(meta, 'keywords', None)}")

## 2. Data Overview
Review available record sets, their fields, and columns. All entity `@id`s are provided for reference and later usage.


In [ ]:
# Find all available record sets in the dataset (using @id). 
record_sets = dataset.record_set_ids()
print(f"Available record sets (@id): {record_sets}")

# For each record set, print field and column @id
for rs_id in record_sets:
    recset = dataset.get_record_set(rs_id)
    print(f"\nRecord Set @id: {recset['@id']}")
    print(f"  Name: {recset.get('name', '')}")
    print(f"  Description: {recset.get('description', '')}")
    # List fields
    fields = recset.get('field', [])
    if isinstance(fields, dict): # only one field
        fields = [fields]
    print("  Fields (@id):")
    for field in fields:
        field_obj = dataset.get_field(field['@id'])
        print(f"    - {field_obj['@id']}: {field_obj.get('name', '')} (type: {field_obj.get('dataType', '')})")
        if 'column' in field_obj:
            col = field_obj['column']
            if isinstance(col, dict):
                col = [col]
            for column in col:
                print(f"      column: {column['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Prepare to extract data from primary record sets. You may want to replace record_set_id with the relevant @id found above.
record_set_ids = dataset.record_set_ids()
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rsid])} records from record set '@id': {rsid}")

# Choose the first available record set for exploration
if record_set_ids:
    chosen_record_set = record_set_ids[0]
    print(f"\nDataFrame columns for record set '@id'={chosen_record_set}:")
    print(dataframes[chosen_record_set].columns.tolist())
    display(dataframes[chosen_record_set].head())
else:
    print('No record sets found in the dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, and grouping data. This section demonstrates operations such as removing outliers, transforming data, or grouping by attributes to prepare for further analysis.


In [ ]:
# For demonstration, select a numeric field for analysis
# You may need to adjust the field names below to match available columns (see previous cell outputs)
import numpy as np

record_set_id = chosen_record_set
df = dataframes[record_set_id]

# Try to guess the first numeric column
numeric_field = None
for col in df.columns:
    # Try converting to numeric sample
    if pd.to_numeric(df[col], errors='coerce').notna().sum() > 0:
        numeric_field = col
        break

if numeric_field is None:
    print('No numeric field available for EDA.')
else:
    print(f"Using numeric field: {numeric_field}")
    # Convert to numeric type safely
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to select a categorical/group field
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].nunique() > 1 and df[col].nunique() < 20:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the mass spectrometry dataset using `mlcroissant`. We listed available record sets and fields (referenced by their `@id`), extracted tabular records, performed basic filtering, normalization, grouping, and created visualizations of the data distribution. This approach can be extended for deeper data analyses, feature engineering, and model development.

For more details, revisit the Croissant schema:
- Croissant schema URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json